# A LoKI session

The local application of the architecture sketch is a notebook on the client
interface (D9). This one tells one session on LoKI@Larmor: find the beam centre,
pick a background run, reduce to I(Q), change the Q binning interactively, fork
the plot, and read the provenance back.

Every step below is the sketch's vocabulary: a **record** is a request plus what
happened to it, a **reference** is how a request names data, a **label** is the
slot an interactive tool owns, the **picker** is the query behind an input field,
a **template** sets a pipeline's parameters and leaves **blanks** for each run to
fill, and a **stage** is the part of the pipeline from the blanks a person moves
to the outputs, which the session holds between runs.

## The client

`local` puts client, backend, launcher, session, and data store in this process.
The specs are bound in-process, which local mode allows as long as no installed
package claims the same name and version. The `FolderSource` is the dataset
source (D7): the folder esssans caches the tutorial files in. Their names are
`<run>-<date>.nxs`, so the source is told the shape of the run identity and the
instrument the names do not carry.

In [ ]:
import tempfile
import time
from pathlib import Path

import ipywidgets as widgets
import plopp as pp
from IPython.display import display

from ess.apps import loki
from ess.apps.client import local
from ess.apps.records import Template
from ess.apps.sources import FolderSource
from ess.apps.spec import DatasetRef, dataset_ref
from ess.reduce.spec.parameters import QEdges

cache = loki.cache()
client = local(
    Path(tempfile.mkdtemp(prefix='loki-session-')),
    instrument='loki',
    proposal='p1',
    submitter='notebook',
    registry=loki.registry(),
    sources=[FolderSource(cache, identity=r'(?P<run>\d+)-.*', instrument='loki')],
)
cache

## The picker

`client.pick()` is what an input field asks: the datasets of every source and the
outputs of completed records, as rows of one shape. Nothing is stored to make the
list. A file whose name carries a run number is identified by instrument and run,
which is what a PID is minted from; the direct-beam file carries none, so its
path is its identity.

In [ ]:
for candidate in client.pick():
    print(f'{candidate.ref!s:<28} {candidate.display["name"]}')

In [ ]:
# Pick the background run by its run number, and the rest of the LoKI@Larmor set.
background = dataset_ref(instrument='loki', run=60393)
inputs = {
    'sample_run': dataset_ref(instrument='loki', run=60387),          # AgBeh
    'sample_transmission_run': dataset_ref(instrument='loki', run=60386),
    'background_run': background,
    'background_transmission_run': dataset_ref(instrument='loki', run=60392),
    'empty_beam_run': dataset_ref(instrument='loki', run=60392),
    'direct_beam': dataset_ref(path=cache / 'direct-beam-loki-all-pixels.h5'),
}
background

## A record: the beam centre

The beam centre is its own spec because it is its own run: one sample run in, one
small value out. The output is a `Quantity`, a vocabulary value, so it is stored
in the record itself rather than in the data store.

In [ ]:
center = client.run(loki.BEAM_CENTER, {'sample_run': inputs['sample_run']})
print(center.id, center.status.value, center.spec)
print('params:', center.request.params)
print('output:', center.outputs['center'])

## A reference: I(Q) from the beam centre

`center.ref()` is "output `center` of record `center.id`". It goes into the
request's parameters as a reference and stays there, which is what makes provenance
the graph you get by following references. The `beam_center` parameter is a union
of a literal and a reference, so a user may equally well type a vector in.

`reduction` is a template that sets every parameter of the I(Q) reduction except
the Q binning, `q`, which the next section moves on a slider.
`reduction.cut(blanks=('q',), name='iofq')` names the part of the pipeline that
`q` feeds, and each `client.run` of it is one record under the label `iofq`: the
parameters of `reduction`, defaults filled, plus the value of `q`. The first call
computes everything.

In [ ]:
def q_edges(num_bins):
    return QEdges(start=0.01, stop=0.3, num_bins=num_bins)


reduction = Template(spec=loki.IOFQ, params=inputs | {'beam_center': center.ref()})
tuning = reduction.cut(blanks=('q',), name='iofq')

started = time.perf_counter()
iofq = client.run(tuning, {'q': q_edges(100)})
print(f'{iofq.status.value} in {time.perf_counter() - started:.2f} s, '
      f'reused={iofq.reused}')
pp.plot(client.output(iofq, 'iofq'), norm='log')

## A slot: the Q binning on a slider

The session built the stage over `q` on the first call and holds it. Everything
the Q binning cannot affect -- loading, masking, the wavelength and Q
conversions, the transmission and direct-beam normalisation -- is held at the
stage's frontier, and a rerun only histograms and subtracts. Which parameters a
stage takes is the caller's decision, not the spec's and not the binding's: a
slider on a different parameter leaves that one unset and names a stage over it.

Each slider move is one call of the stage under the label `iofq`. That label is
the slot the plot owns: the newest record under it supersedes the earlier ones,
and none of them is lost. Every record is complete as written, so recomputing
one needs nothing from the session.

`record.reused` says that the result came out of a stage the session was already
holding; the time per rerun is the other sign of it.

In [ ]:
log = []
slider = widgets.IntSlider(value=100, min=25, max=300, step=25, description='Q bins')


def rebin(change):
    started = time.perf_counter()
    record = client.run(tuning, {'q': q_edges(change['new'])})
    log.append(
        f'q_bins={change["new"]:>3}  {record.id}  reused={record.reused}  '
        f'{time.perf_counter() - started:.2f} s'
    )
    display(pp.plot(client.output(record, 'iofq'), norm='log'))


slider.observe(rebin, names='value')
slider

In [ ]:
# Executed headless, these stand in for dragging the slider.
for value in (200, 50, 300):
    slider.value = value
print('\n'.join(log))

## The slot's records

`client.latest` answers the one query the backend offers for a label. `batch` is
the latest record per member key, which for a slider with no member key is that
same record; the full history under the label is an ordinary record query.

In [ ]:
print('latest:', client.latest('iofq').id)
print('batch: ', [r.id for r in client.batch('iofq')])
for record in client.records(label='iofq'):
    print(record.id, record.request.params['q']['num_bins'], record.status.value)

## Forking the slot

Comparing two variants side by side is two labels. The second is assigned when
the user forks: the same stage under another name, `tuning.cut(name='iofq-fine')`,
whose records carry that name as their label. The session holds stages by what
they are, not by label, so the fork comes out of
the stage the slider already built. Discarding a variant drops its label from
the UI and nothing else.

In [ ]:
fine = client.run(tuning.cut(name='iofq-fine'), {'q': q_edges(300)})
coarse = client.latest('iofq')
print('fork reused:', fine.reused)
display(widgets.HBox([
    pp.plot(client.output(coarse, 'iofq'), norm='log', title='iofq').to_widget(),
    pp.plot(client.output(fine, 'iofq'), norm='log', title='iofq-fine').to_widget(),
]))

## Provenance

The provenance snapshot of a record names its spec, its parameters, and the
outputs it computed. Following the references of the latest I(Q) record
reaches the beam-centre record and, through both, the dataset references the
requests name. Datasets are the leaves: they have no record and nothing to
recompute.

In [ ]:
provenance = client.provenance(client.latest('iofq'))
print(provenance['spec'], provenance['package_versions'], provenance['binding'])
print('outputs:', provenance['outputs'])
print('datasets:', [str(DatasetRef(**raw)) for raw in provenance['raw']])
for upstream in provenance['inputs']:
    print('from', upstream['spec'], upstream['record'],
          'over', [str(DatasetRef(**raw)) for raw in upstream['raw']])